In [16]:
import sys
from pathlib import Path
import h5py
import numpy as np

# Ensure Python sees Kosmulator's modules
sys.path.append(str(Path(".").resolve()))

# Import your existing post-processing functions directly
# (replace these names with your exact function names from Kosmulator)
from Kosmulator_main import Post_processing

In [17]:
from pathlib import Path
from typing import Dict, Any, Optional
import h5py
import numpy as np


def extract_chain_data(
    file_path: Path, burn_fraction: float = 0.3, min_steps: int = 1000
) -> Optional[Dict[str, Any]]:
    """Extracts valid completed steps from an .h5 chain, discarding incomplete runs."""
    try:
        with h5py.File(file_path, "r") as f:
            if "mcmc" not in f:
                return None

            # 1. Determine actual completed iterations
            iteration = f["mcmc"].attrs.get("iteration", None)
            if iteration is None:
                if "accepted" in f["mcmc"]:
                    iteration = int(np.max(f["mcmc/accepted"][()]))
                else:
                    lp = f["mcmc/log_prob"][:, 0]
                    nz = np.where(lp != 0.0)[0]
                    iteration = int(nz[-1] + 1) if len(nz) > 0 else 0

            # Reject crashed/aborted runs early
            if iteration < min_steps:
                print(
                    f"  [SKIP] {file_path.stem} only completed {iteration} steps (< {min_steps})."
                )
                return None

            # 2. Slice strictly up to the completed iteration
            chain = f["mcmc/chain"][:iteration]
            log_prob = f["mcmc/log_prob"][:iteration]
            attrs = {k: v for k, v in f.attrs.items()}

        # 3. Apply burn-in to the COMPLETED slice
        burn_idx = int(burn_fraction * iteration)
        trimmed_chain = chain[burn_idx:]
        trimmed_log_prob = log_prob[burn_idx:]

        flat_samples = trimmed_chain.reshape(-1, trimmed_chain.shape[-1])
        flat_log_prob = trimmed_log_prob.reshape(-1)

        return {
            "raw_chain": trimmed_chain,
            "raw_log_prob": trimmed_log_prob,
            "flat_samples": flat_samples,
            "flat_log_prob": flat_log_prob,
            "shape": trimmed_chain.shape,
            "completed_steps": iteration,
            "attrs": attrs,
        }

    except Exception as e:
        print(f"[ERROR] Failed reading {file_path.name}: {e}")
        return None


def extract_all_marcel_chains(
    base_dir: str = "MCMC_Chains/Marcel_Chains", 
    burn_fraction: float = 0.0
) -> Dict[str, Dict[str, Any]]:
    """
    Recursively scans the directory for all .h5 files, ignoring metadata artifacts,
    and indexes them by their hierarchical path tags.
    """
    root = Path(base_dir)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root.resolve()}")

    extracted_data = {}

    # Gather all .h5 files (excluding Windows alternate data stream Zone.Identifier files)
    all_h5 = [
        p for p in root.rglob("*.h5") 
        if not p.name.endswith(".Zone.Identifier")
    ]

    print(f"Found {len(all_h5)} valid .h5 chain files under {root}.\n")

    for file_path in sorted(all_h5):
        # Extract relative path components: e.g. Free / NonLinear_IDE_2 / combination
        rel_parts = file_path.relative_to(root).parts
        
        # Build an identifier label: "Free__NonLinear_IDE_2__DESI_DR2_PantheonP_SH0ES"
        chain_key = "__".join(rel_parts[:-1]) if len(rel_parts) > 1 else file_path.stem
        
        print(f"Loading: {chain_key} ({file_path.name})...")
        data = extract_chain_data(file_path, burn_fraction=burn_fraction)

        if data is not None:
            data["path"] = str(file_path)
            data["regime"] = rel_parts[0] if len(rel_parts) > 0 else "unknown"
            data["model"] = rel_parts[1] if len(rel_parts) > 1 else "unknown"
            data["dataset_combo"] = rel_parts[2] if len(rel_parts) > 2 else file_path.stem
            
            extracted_data[chain_key] = data
            print(f"  └── Success: shape={data['shape']}, total flat samples={len(data['flat_samples'])}")

    return extracted_data


if __name__ == "__main__":
    # Test execution
    chains = extract_all_marcel_chains(burn_fraction=0.2)
    print(f"\nSuccessfully indexed {len(chains)} chains ready for statistical analysis.")

all_chains = extract_all_marcel_chains(
    base_dir="MCMC_Chains/Marcel_Chains", 
    burn_fraction=0.3
)

Found 9 valid .h5 chain files under MCMC_Chains/Marcel_Chains.

Loading: Free__LCDM_v__BBN_PryMordial_CC_DESI_DR2_Pantheon (BBN_PryMordial+CC+DESI_DR2+Pantheon.h5)...
  [SKIP] BBN_PryMordial+CC+DESI_DR2+Pantheon only completed 149 steps (< 1000).
Loading: Free__LCDM_v__BBN_PryMordial_CC_DESI_DR2_PantheonP_SH0ES (BBN_PryMordial+CC+DESI_DR2+PantheonP_SH0ES.h5)...
  [SKIP] BBN_PryMordial+CC+DESI_DR2+PantheonP_SH0ES only completed 4 steps (< 1000).
Loading: Free__LCDM_v__DESI_DR2_PantheonP_SH0ES (DESI_DR2+PantheonP_SH0ES.h5)...
  └── Success: shape=(12000, 120, 4), total flat samples=1440000
Loading: Free__NonLinear_IDE_2__BBN_PryMordial_CC_DESI_DR2_Pantheon (BBN_PryMordial+CC+DESI_DR2+Pantheon.h5)...
  [SKIP] BBN_PryMordial+CC+DESI_DR2+Pantheon only completed 3 steps (< 1000).
Loading: Free__NonLinear_IDE_2__DESI_DR2_PantheonP_SH0ES (DESI_DR2+PantheonP_SH0ES.h5)...
  └── Success: shape=(21840, 120, 6), total flat samples=2620800
Loading: Positive__LCDM_v__DESI_DR2_PantheonP_SH0ES (DESI_DR

In [18]:
# Group extracted chains by (model, observation_combo)
# If comparing against LCDM_v, you can group them under their respective model names
grouped_chains = {}

for chain_key, chain_data in all_chains.items():
    model = chain_data["model"]
    obs = chain_data["dataset_combo"]
    regime = chain_data["regime"]
    
    # Optional: tag model name with regime if analyzing Free vs Positive vs Positive_stable
    full_model_tag = f"{model}_{regime}" if regime != "unknown" else model
    
    if full_model_tag not in grouped_chains:
        grouped_chains[full_model_tag] = {}
        
    grouped_chains[full_model_tag][obs] = {
        "samples": chain_data["flat_samples"],
        "loglike": chain_data["flat_log_prob"],
    }

print(f"Grouped models: {list(grouped_chains.keys())}")

Grouped models: ['LCDM_v_Free', 'NonLinear_IDE_2_Free', 'LCDM_v_Positive', 'NonLinear_IDE_2_Positive', 'LCDM_v_Positive_stable', 'NonLinear_IDE_2_Positive_stable']


In [19]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from Kosmulator_main import Post_processing as PP

PARAM_NAMES = {
    "LCDM_v": {
        "BBN_PryMordial_CC_DESI_DR2_Pantheon":        ["Omega_m", "Omega_bh^2", "H_0"],
        "BBN_PryMordial_CC_DESI_DR2_PantheonP_SH0ES": ["Omega_m", "Omega_bh^2", "H_0", "M_abs"],
        "DESI_DR2_PantheonP_SH0ES":                   ["Omega_m", "H_0", "r_d", "M_abs"],
    },
    "NonLinear_IDE_2": {
        "BBN_PryMordial_CC_DESI_DR2_Pantheon":        ["Omega_m", "w", "delta", "Omega_bh^2", "H_0"],
        "DESI_DR2_PantheonP_SH0ES":                   ["Omega_m", "w", "delta", "H_0", "r_d", "M_abs"],
    },
}

OBS_LISTS = {
    "BBN_PryMordial_CC_DESI_DR2_Pantheon": [
        "BBN_PryMordial",
        "CC",
        "DESI_DR2",
        "Pantheon",
    ],
    # Check whether your Kosmulator likelihood is named PantheonP_SH0ES or PantheonPS
    "BBN_PryMordial_CC_DESI_DR2_PantheonP_SH0ES": [
        "BBN_PryMordial",
        "CC",
        "DESI_DR2",
        "PantheonPS",
    ],
    "DESI_DR2_PantheonP_SH0ES": ["DESI_DR2", "PantheonPS"],
}

all_structured_values = {}
all_latex_tables = {}

for model_tag, obs_dict in grouped_chains.items():
    base_model = model_tag.split("_Free")[0].split("_Positive")[0]
    all_structured_values[model_tag] = {}
    all_latex_tables[model_tag] = {}

    for obs_key, chain_data in obs_dict.items():
        params = PARAM_NAMES.get(base_model, {}).get(obs_key)
        comp_list = OBS_LISTS.get(obs_key)

        if params is None or comp_list is None:
            print(f"[SKIP] No mapping for {base_model} / {obs_key}")
            continue

        # In case comp_list was wrapped as [[...]], extract the 1D list
        if isinstance(comp_list[0], list):
            comp_list = comp_list[0]

        # Use the joined name that matches the component list if Kosmulator expects that
        sample_key = "_".join(comp_list)

        results, latex_table, structured_values = (
            PP.calculate_asymmetric_from_samples(
                samples={sample_key: chain_data},
                parameters=[params],  # List of lists -> parameters[0] = params
                observations=[
                    comp_list
                ],  # List of component lists -> observations[0] = comp_list
            )
        )

        all_structured_values[model_tag][obs_key] = structured_values
        all_latex_tables[model_tag][obs_key] = latex_table
        print(f"Done: {model_tag} / {obs_key}")


--- Diagnostic: DESI_DR2_PantheonPS ---
obs_samples type: <class 'numpy.ndarray'>, shape: (1260000, 4)
log_like type:    <class 'numpy.ndarray'>, shape: (1260000,)
log_like ndim:    1
log_like sample values: [-735.26440859 -734.26277935 -734.05660541]
Done: LCDM_v_Free / DESI_DR2_PantheonP_SH0ES

--- Diagnostic: DESI_DR2_PantheonPS ---
obs_samples type: <class 'numpy.ndarray'>, shape: (2293200, 6)
log_like type:    <class 'numpy.ndarray'>, shape: (2293200,)
log_like ndim:    1
log_like sample values: [-734.81188387 -731.39189251 -735.11255874]
Done: NonLinear_IDE_2_Free / DESI_DR2_PantheonP_SH0ES

--- Diagnostic: DESI_DR2_PantheonPS ---
obs_samples type: <class 'numpy.ndarray'>, shape: (1243200, 4)
log_like type:    <class 'numpy.ndarray'>, shape: (1243200,)
log_like ndim:    1
log_like sample values: [-734.52721097 -734.80687934 -733.97187444]
Done: LCDM_v_Positive / DESI_DR2_PantheonP_SH0ES

--- Diagnostic: DESI_DR2_PantheonPS ---
obs_samples type: <class 'numpy.ndarray'>, shape: (1

In [20]:
#Diagnostic test to see the nature of loglikes
import h5py
import numpy as np
from pathlib import Path

# Pick any one of your .h5 chain files
sample_file = next(Path("MCMC_Chains/Marcel_Chains").rglob("*.h5"))
print(f"Inspecting file: {sample_file}\n")


def inspect_node(name, obj):
    if isinstance(obj, h5py.Dataset):
        data_preview = obj[()]
        min_val = np.min(data_preview) if data_preview.size > 0 else "empty"
        max_val = np.max(data_preview) if data_preview.size > 0 else "empty"
        non_zeros = (
            np.count_nonzero(data_preview) if data_preview.size > 0 else 0
        )
        print(
            f"Dataset: {name:<25} shape={str(obj.shape):<18} dtype={str(obj.dtype):<10} "
            f"range=[{min_val}, {max_val}] non_zero={non_zeros}/{data_preview.size}"
        )


with h5py.File(sample_file, "r") as f:
    f.visititems(inspect_node)

Inspecting file: MCMC_Chains/Marcel_Chains/Free/NonLinear_IDE_2/BBN_PryMordial_CC_DESI_DR2_Pantheon/BBN_PryMordial+CC+DESI_DR2+Pantheon.h5

Dataset: mcmc/accepted             shape=(128,)             dtype=float64    range=[0.0, 2.0] non_zero=72/128
Dataset: mcmc/blobs                shape=(100000, 128)      dtype=float64    range=[-9913270.706741735, 0.0] non_zero=384/12800000
Dataset: mcmc/chain                shape=(100000, 128, 5)   dtype=float64    range=[-2.0, 54.94777881309241] non_zero=1920/64000000
Dataset: mcmc/log_prob             shape=(100000, 128)      dtype=float64    range=[-9913270.706741735, 0.0] non_zero=384/12800000


In [21]:
from pathlib import Path
import h5py
import numpy as np

base_path = Path("MCMC_Chains/Marcel_Chains")
all_files = sorted(
    [
        p
        for p in base_path.rglob("*.h5")
        if not p.name.endswith(".Zone.Identifier")
    ]
)

print(f"{'Chain Identifier':<60} | {'Iter':<6} | {'Allocated':<9} | {'Status'}")
print("-" * 90)

usable_chains = []

for file_path in all_files:
    rel_name = "__".join(file_path.relative_to(base_path).parts[:-1])
    if not rel_name:
        rel_name = file_path.stem

    try:
        with h5py.File(file_path, "r") as f:
            if "mcmc" not in f:
                print(f"{rel_name:<60} | {'N/A':<6} | {'N/A':<9} | Missing 'mcmc' group")
                continue

            # 1. emcee stores the true step count in backend attributes
            iteration = f["mcmc"].attrs.get("iteration", None)
            total_allocated = f["mcmc/chain"].shape[0]

            # 2. Fallback check: look at max accepted steps or non-zero entries in log_prob
            if iteration is None:
                if "accepted" in f["mcmc"]:
                    iteration = int(np.max(f["mcmc/accepted"][()]))
                else:
                    # Probe log_prob backwards to find where non-zeros end
                    lp_sample = f["mcmc/log_prob"][:, 0]
                    nonzero_idx = np.where(lp_sample != 0.0)[0]
                    iteration = int(nonzero_idx[-1] + 1) if len(nonzero_idx) > 0 else 0

            # Determine usable status
            # A run needs enough completed steps to survive burn-in (e.g. at least 1,000 steps)
            if iteration < 50:
                status = f"FAILED / INCOMPLETE ({iteration} steps completed)"
            elif iteration < total_allocated:
                status = f"PARTIAL ({iteration}/{total_allocated} steps)"
                usable_chains.append((rel_name, file_path, iteration))
            else:
                status = f"COMPLETE ({iteration} steps)"
                usable_chains.append((rel_name, file_path, iteration))

            print(f"{rel_name:<60} | {iteration:<6} | {total_allocated:<9} | {status}")

    except Exception as e:
        print(f"{rel_name:<60} | {'ERR':<6} | {'ERR':<9} | Read error: {e}")

print("\n" + "=" * 90)
print(f"Total usable chains found: {len(usable_chains)} of {len(all_files)}")

Chain Identifier                                             | Iter   | Allocated | Status
------------------------------------------------------------------------------------------
Free__LCDM_v__BBN_PryMordial_CC_DESI_DR2_Pantheon            | 149    | 100000    | PARTIAL (149/100000 steps)
Free__LCDM_v__BBN_PryMordial_CC_DESI_DR2_PantheonP_SH0ES     | 4      | 100000    | FAILED / INCOMPLETE (4 steps completed)
Free__LCDM_v__DESI_DR2_PantheonP_SH0ES                       | 15000  | 100000    | PARTIAL (15000/100000 steps)
Free__NonLinear_IDE_2__BBN_PryMordial_CC_DESI_DR2_Pantheon   | 3      | 100000    | FAILED / INCOMPLETE (3 steps completed)
Free__NonLinear_IDE_2__DESI_DR2_PantheonP_SH0ES              | 27300  | 100000    | PARTIAL (27300/100000 steps)
Positive__LCDM_v__DESI_DR2_PantheonP_SH0ES                   | 14800  | 100000    | PARTIAL (14800/100000 steps)
Positive__NonLinear_IDE_2__DESI_DR2_PantheonP_SH0ES          | 19000  | 100000    | PARTIAL (19000/100000 steps)
Positiv

In [26]:
#Running the statistical analysis for the correct chains:
import numpy as np
import pandas as pd
from scipy.stats import chi2, norm

# Total independent data points for DESI DR2 + Pantheon+ / SH0ES
# Pantheon+ & SH0ES: 1701 SNe (calibrated distance moduli)
# DESI DR2 BAO: ~14 to 18 effective bins depending on redshift slices
# Adjust N_DATA if your exact config includes specific bin counts

#Calculating significance:
def significance(dchi, r):
    """
    p - tail probability
    sigma - equivalent gaussian significance
    """
    dchi = np.abs(dchi)
    r = np.round(r, decimals = 0)
    if np.isnan(dchi) or np.isnan(r):
        sigma = 0
    else:
        p = chi2.sf(dchi, df = r)
        sigma = norm.isf(p/2)
    return np.round(sigma, decimals = 1)


N_DATA_MAP = {
    "DESI_DR2_PantheonPS": 1701 + 14,
    "BBN_PryMordial_CC_DESI_DR2_Pantheon": 1048 + 32 + 14 + 1,
}


def compute_ic_table(grouped_chains, reference_model_tag="LCDM_v_Free"):
    records = []

    for model_tag, obs_dict in grouped_chains.items():
        for obs_key, chain_data in obs_dict.items():
            samples = chain_data["samples"]
            loglike = chain_data["loglike"]

            if loglike is None or len(loglike) == 0:
                continue

            k = samples.shape[1]  # Number of parameters
            N = N_DATA_MAP.get(
                obs_key, 1715
            )  # Default N if not explicitly mapped

            # Deviance D(theta) = -2 ln(L)
            deviances = -2.0 * loglike
            d_min = float(np.min(deviances))  # D_hat (Maximum a Posteriori) Is also chi_min
            d_bar = float(np.mean(deviances))  # Mean deviance

            # Effective number of parameters p_D
            # Spiegelhalter definition: p_D = D_bar - D_hat
            p_D = d_bar - d_min
            # Gelman variance definition alternative: p_V = 0.5 * Var(D)
            p_V = 0.5 * np.var(deviances, ddof=1)

            dic = d_bar + p_D
            aic = d_min + 2 * k
            bic = d_min + k * np.log(N)
            

            # AICc correction
            if (N - k - 1) > 0:
                aicc = aic + (2 * k * (k + 1)) / (N - k - 1)
            else:
                aicc = aic

            records.append(
                {
                    "Model": model_tag,
                    "Dataset": obs_key,
                    "k": k,
                    "N": N,
                    "chi2_min": d_min,
                    "p_D": p_D,
                    "AIC": aic,
                    "AICc": aicc,
                    "BIC": bic,
                    "DIC": dic,
                }
            )

    df = pd.DataFrame(records)

    # Compute Deltas relative to reference model
    ref_rows = df[df["Model"] == reference_model_tag].set_index("Dataset")

    df["dAIC"] = df.apply(
        lambda r: r["AIC"] - ref_rows.loc[r["Dataset"], "AIC"]
        if r["Dataset"] in ref_rows.index
        else np.nan,
        axis=1,
    )
    df["dAICc"] = df.apply(
        lambda r: r["AICc"] - ref_rows.loc[r["Dataset"], "AICc"]
        if r["Dataset"] in ref_rows.index
        else np.nan,
        axis=1,
    )
    df["dBIC"] = df.apply(
        lambda r: r["BIC"] - ref_rows.loc[r["Dataset"], "BIC"]
        if r["Dataset"] in ref_rows.index
        else np.nan,
        axis=1,
    )
    df["dDIC"] = df.apply(
        lambda r: r["DIC"] - ref_rows.loc[r["Dataset"], "DIC"]
        if r["Dataset"] in ref_rows.index
        else np.nan,
        axis=1,
    )
    df['dChi'] = df.apply(
        lambda r: r['chi2_min'] - ref_rows.loc[r["Dataset"], 'chi2_min']
        if r['Dataset'] in ref_rows.index
        else np.nan,
        axis = 1,
    )
    df['Significance'] = df.apply(
        lambda r: significance(r['dChi'], (r['p_D'] - ref_rows.loc[r["Dataset"], 'p_D']))
        if r['Dataset'] in ref_rows.index
        else np.nan,
        axis = 1
    )

    return df.sort_values(by=["Dataset", "dDIC"])


# Run comparison table
ic_summary = compute_ic_table(grouped_chains, reference_model_tag="LCDM_v_Free")
ic_summary.round(3)

,Model,Dataset,k,N,chi2_min,p_D,AIC,AICc,BIC,DIC,dAIC,dAICc,dBIC,dDIC,dChi,Significance
1,NonLinear_IDE_2_Free,DESI_DR2_PantheonP_SH0ES,6,1715,1460.678,6.008,1472.678,1472.728,1505.361,1472.694,-0.850,-0.824,10.044,-0.827,-4.850,1.7
0,LCDM_v_Free,DESI_DR2_PantheonP_SH0ES,4,1715,1465.529,3.996,1473.529,1473.552,1495.317,1473.521,0.000,0.000,0.000,0.000,0.000,NaN
2,LCDM_v_Positive,DESI_DR2_PantheonP_SH0ES,4,1715,1465.528,3.997,1473.528,1473.551,1495.316,1473.522,-0.001,-0.001,-0.001,0.001,-0.001,NaN
4,LCDM_v_Positive_stable,DESI_DR2_PantheonP_SH0ES,4,1715,1465.530,4.006,1473.530,1473.553,1495.319,1473.542,0.001,0.001,0.001,0.021,0.001,NaN
3,NonLinear_IDE_2_Positive,DESI_DR2_PantheonP_SH0ES,6,1715,1461.605,6.551,1473.605,1473.654,1506.288,1474.707,0.076,0.102,10.970,1.185,-3.924,1.1
5,NonLinear_IDE_2_Positive_stable,DESI_DR2_PantheonP_SH0ES,6,1715,1463.136,6.670,1475.136,1475.185,1507.819,1476.476,1.607,1.633,12.501,2.955,-2.393,0.7


In [27]:
#Clean data:
export_df = ic_summary.copy()

# Replace NaNs in significance with dashes or 0.0
export_df["Significance"] = export_df["Significance"].fillna(0.0)

# Clean and reorder columns for publication layout
cols_order = [
    "Model",
    "Dataset",
    "k",
    "N",
    "chi2_min",
    "p_D",
    "AIC",
    "dAIC",
    "AICc",
    "dAICc",
    "BIC",
    "dBIC",
    "DIC",
    "dDIC",
    "dChi",
    "Significance",
]
export_df = export_df[[c for c in cols_order if c in export_df.columns]]

In [28]:
#Generating a latex table:
# Generate clean LaTeX code
latex_str = export_df.to_latex(
    index=False,
    float_format="%.3f",
    column_format="llcccccccccccccc",
    escape=False,  # Allows math characters like \Delta if you rename columns
)

with open("model_comparison_statistics.tex", "w") as f:
    f.write(latex_str)

In [29]:
#Export to csv and excel
# CSV (plain text)
export_df.to_csv(
    "model_comparison_statistics.csv", index=False, float_format="%.3f"
)

# Excel workbook (requires openpyxl)
export_df.to_excel("model_comparison_statistics.xlsx", index=False)